<a href="https://colab.research.google.com/github/nurfnick/Mathematical_Musings/blob/main/LetterGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import re
from pathlib import Path

# ------------------------------------------------
# FILE LOCATIONS
# ------------------------------------------------

excel_file = "FinalReceived.xlsx"
template_file = "template.tex"

output_folder = Path("Donation_Letters")
output_folder.mkdir(exist_ok=True)

# Set to True if you have pdflatex installed
COMPILE_PDF = False


# ------------------------------------------------
# LATEX FORMATTING
# ------------------------------------------------

def latex_escape(value):
    """Escape special LaTeX characters in donor names."""

    value = str(value)

    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    for char, replacement in replacements.items():
        value = value.replace(char, replacement)

    return value


def replace_command(template, command, value):
    """Replace the contents of a LaTeX newcommand."""

    pattern = (
        r"(\\newcommand\{\\"
        + re.escape(command)
        + r"\}\{).*?(\})"
    )

    return re.sub(
        pattern,
        lambda m: m.group(1) + value + m.group(2),
        template,
        count=1,
        flags=re.DOTALL
    )


# ------------------------------------------------
# READ EXCEL FILE AND LATEX TEMPLATE
# ------------------------------------------------

df = pd.read_excel(excel_file)

with open(template_file, "r", encoding="utf-8") as file:
    original_template = file.read()

# Check that required columns exist
required_columns = ["Name", "Total Donation"]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(
            f"Missing required Excel column: {column}"
        )


# ------------------------------------------------
# GENERATE A LETTER FOR EACH DONOR
# ------------------------------------------------

letters_created = 0
donors_skipped = 0

for index, row in df.iterrows():

    # Skip rows without a donor name
    if pd.isna(row["Name"]):
        donors_skipped += 1
        continue

    # Convert donation amount to a number
    amount = pd.to_numeric(
        str(row["Total Donation"]).replace("$", "").replace(",", ""),
        errors="coerce"
    )

    # Skip blank or invalid donation amounts
    if pd.isna(amount):
        donors_skipped += 1
        continue

    # Skip zero donations
    if amount == 0:
        donors_skipped += 1
        continue

    donor = latex_escape(row["Name"])
    amount = f"{amount:,.2f}"

    # Start with a fresh copy of the template
    letter = original_template

    # Replace donor-specific commands
    letter = replace_command(letter, "donor", donor)
    letter = replace_command(letter, "amount", amount)

    # Remove mailing address from the letter environment
    letter = re.sub(
        r"(\\begin\{letter\})\{.*?\}",
        r"\1{}",
        letter,
        count=1,
        flags=re.DOTALL
    )

    # Create a safe filename
    filename = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(row["Name"])
    ).strip("_")

    output_file = output_folder / f"{filename}.tex"

    # Save the LaTeX file
    with open(output_file, "w", encoding="utf-8") as file:
        file.write(letter)

    letters_created += 1

    print(f"Created: {output_file} (${amount})")


print("\n--------------------------------")
print("DONATION LETTER GENERATION")
print("--------------------------------")
print(f"Letters created: {letters_created}")
print(f"Donors skipped:  {donors_skipped}")
print("--------------------------------")


# ------------------------------------------------
# OPTIONAL: COMPILE LETTERS INTO PDF FILES
# ------------------------------------------------

if COMPILE_PDF:

    import subprocess
    import shutil

    for tex_file in output_folder.glob("*.tex"):

        source_dir = Path(template_file).resolve().parent

        # Copy graphics needed by the template
        for asset in [
            "miz-rugby.png",
            "signature.pdf",
            "signature.png",
            "signature.jpg"
        ]:
            source = source_dir / asset

            if source.exists():
                shutil.copy2(
                    source,
                    output_folder / asset
                )

        subprocess.run(
            [
                "pdflatex",
                "-interaction=nonstopmode",
                tex_file.name
            ],
            cwd=output_folder,
            check=True
        )

        print(f"Compiled PDF: {tex_file.stem}.pdf")


Created: Donation_Letters/Ben_Harrison.tex ($1,900.00)
Created: Donation_Letters/Billy_Herbert.tex ($900.00)
Created: Donation_Letters/Cherrly_Dillion.tex ($50.00)
Created: Donation_Letters/Chuck_Bruder.tex ($600.00)
Created: Donation_Letters/Don_Corwin.tex ($500.00)
Created: Donation_Letters/Ed_Reed.tex ($1,300.00)
Created: Donation_Letters/Ellen_Ermeling.tex ($50.00)
Created: Donation_Letters/Jacob_Reuter.tex ($25.00)
Created: Donation_Letters/Jim_Dierker.tex ($400.00)
Created: Donation_Letters/John_McGuire.tex ($900.00)
Created: Donation_Letters/John_Tarpoff.tex ($2,900.00)
Created: Donation_Letters/John_Victor.tex ($1,525.00)
Created: Donation_Letters/Julie_Devoti.tex ($100.00)
Created: Donation_Letters/Kevin_Baniak.tex ($125.00)
Created: Donation_Letters/Kyle_Pusateri.tex ($100.00)
Created: Donation_Letters/Matt_Brotemarkle.tex ($25.00)
Created: Donation_Letters/Nicholas_Allen.tex ($50.00)
Created: Donation_Letters/Nicholas_Floppy_Jacob.tex ($645.00)
Created: Donation_Letters/Nick